# 18.3 只能看到教師回答也能學嗎？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**人寫題解拿來SFT，能說已經完成teacher distillation嗎？

教師生成答案後保存來源與版本，再按SFT資料訓練學生。先過濾不一致、無法驗證或違反目標的答案；GSM8K人寫步驟不能標成教師生成。

**把直覺寫成數學：**black-box distillation複用assistant-only CE；教師生成成本另算。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
record = {
    "messages": [{"role": "user", "content": "2+2=?"}, {"role": "assistant", "content": "4"}],
    "teacher_revision": "待真實教師生成時填寫",
    "provenance": "人工示例，非教師生成",
}
print(record)
print("教師生成→驗證答案→保存JSONL→train.py --task sft --data ... --train")

**如何讀結果：**只看到答案可以不同tokenizer，因爲先decode成文字再由學生encode；這不同於白盒分佈對齊。

**只改一件事：**只改provenance字段爲真實模型commit，前提是確實執行生成並保存。
